# Phase 12 - Verhalten: kippen verankerte Prompts tatsaechlich seltener?

Alle Anker-Messungen bisher waren **Dispositions**-Messungen (Fremdschrift-Masse an
einer Position). Diese Zelle erzeugt zum ersten Mal Antworten und zaehlt, wie oft die
Sprache tatsaechlich kippt - mit dem Klassifikator aus Cell 29b und einer Baseline,
die im selben Lauf unter demselben Protokoll mitlaeuft.

Neun Arme ersetzen genau die Wortgruppe `each service's local name`:
Ortsanker (`Brazilian` / `Japanese`), Autoritaet (`official`), ein **Placebo** ohne Ort
(`exact`), die aufgeloeste von-Konstruktion, der blosse Artikel, sowie zwei Arme ganz
**ohne** ` local`. Fuenf Vorhersagen der Referenzluecken-Lesart sind vorab registriert
und werden einzeln als HAELT/FAELLT gedruckt.

Selbstversorgend - **frische Laufzeit**, dann nur diese Zelle. Laufzeit ~10-15 min
(9 x 48 Ziehungen x 40 neue Token). Protokoll und JSON gehen direkt nach Drive.


In [ ]:
# === PHASE 12 - VERHALTEN: KIPPEN VERANKERTE PROMPTS SELTENER? =============
# Alle bisherigen Anker-Messungen waren DISPOSITIONS-Messungen: Fremdschrift-
# Masse an einer Position, nie eine erzeugte Antwort. Diese Zelle fragt zum
# ersten Mal das Verhalten selbst - mit dem etablierten Klassifikator aus
# Cell 29b und einer Baseline, die im selben Lauf unter demselben Protokoll
# mitgemessen wird (keine Zahl aus einem anderen Lauf wird verglichen).
#
# Der Rahmen heisst jetzt REFERENZLUECKE, nicht mehr Schrifthypothese:
# "each service's local name" verlangt einen Ort, der Prompt nennt keinen
# (Google Drive, Dropbox, OneDrive sind drei US-Dienste, kein Land). Das Modell
# muss einen Ort liefern und liefert einen nicht-englischen.
#
# NEUN ARME - jeder ersetzt genau die Wortgruppe "each service's local name":
#   original    each service's local name              Luecke offen  (Baseline)
#   latein      each service's Brazilian local name     Ort genannt, lat. Schrift
#   fremd       each service's Japanese local name      Ort genannt, fremde Schrift
#   amtlich     each service's official local name      Autoritaet statt Ort
#   blass       each service's exact local name         PLACEBO: Einfuegung ohne Ort
#   von         the local name of each service          Genitiv aufgeloest
#   artikel     the local name                          Bezug ganz entfernt
#   ohne_local  each service's name                     ist ' local' noetig?
#   fremd_ohne  each service's Japanese name            Anker ohne ' local'
#
# VORAB REGISTRIERTE VORHERSAGEN der Referenzluecken-Lesart (jede einzeln
# falsifizierbar, Urteil wird gedruckt, nicht nachtraeglich gewaehlt):
#   V1  latein  < original     Ort schliesst die Luecke
#   V2  fremd   < original     ebenso - und V1 ~ V2 (Schrift egal, s. Anker v2)
#   V3  blass  ~= original     PLACEBO: blosse Einfuegung reicht NICHT
#   V4  ohne_local < original  ohne ' local' gibt es keine Luecke
#   V5  von    ~= original     der Genitiv traegt sie nicht, das Wort tut es
# Faellt V3, ist der Effekt unspezifisch (jede Einfuegung wirkt). Faellt V4
# nicht, ist ' local' nicht das tragende Element und die Lesart braucht Ersatz.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
import glob, json, gc
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig). "
        "Loesung: Laufzeit -> Sitzung neu starten, dann NUR diese Zelle.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_verhalten")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- Prompt beschaffen -----------------------------------------
ZIEL_ID=globals().get("ZIEL_ID","")            # optional vorbelegt
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
ARME=[("original"  ,"each service's local name"          ,"Luecke offen (Baseline)"),
      ("latein"    ,"each service's Brazilian local name" ,"Ort, lateinische Schrift"),
      ("fremd"     ,"each service's Japanese local name"  ,"Ort, fremde Schrift"),
      ("amtlich"   ,"each service's official local name"  ,"Autoritaet statt Ort"),
      ("blass"     ,"each service's exact local name"     ,"PLACEBO: Einfuegung ohne Ort"),
      ("von"       ,"the local name of each service"      ,"Genitiv aufgeloest"),
      ("artikel"   ,"the local name"                      ,"Bezug entfernt"),
      ("ohne_local","each service's name"                 ,"ohne ' local'"),
      ("fremd_ohne","each service's Japanese name"        ,"Ort ohne ' local'")]
def setze_arm(text,ersatz):
    """genau eine Ersetzung der Zielphrase; (neuer_text, ok)"""
    if text.count(PHRASE)!=1: return text,False
    return text.replace(PHRASE,ersatz),True
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    """identisch zu Cell 29b - damit die Raten vergleichbar bleiben"""
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
SW=("takeover","gloss","latin-switch(fr)")
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def twoprop(k1,n1,k2,n2):
    p=(k1+k2)/(n1+n2); se=math.sqrt(p*(1-p)*(1/n1+1/n2)) if 0<p<1 else 0.0
    if se==0: return 1.0
    z=abs(k1/n1-k2/n2)/se
    return 2*(1-0.5*(1+math.erf(z/math.sqrt(2))))
def faellt(k,n,k0,n0,alpha=0.05):
    """sinkt der Arm signifikant unter die Baseline?"""
    return (k/n < k0/n0) and twoprop(k,n,k0,n0)<alpha
def gleich(k,n,k0,n0,alpha=0.05):
    """kein nachweisbarer Unterschied zur Baseline"""
    return twoprop(k,n,k0,n0)>=alpha
def pruefe_vorhersagen(K,N):
    """K: dict arm->Treffer, N: dict arm->Ziehungen. Liste (name,text,ok)"""
    b=lambda a:(K[a],N[a]); k0,n0=b("original")
    V=[]
    V.append(("V1","latein < original",      faellt(*b("latein"),k0,n0)))
    V.append(("V2","fremd  < original",      faellt(*b("fremd"), k0,n0)))
    V.append(("V2b","latein ~= fremd",       gleich(*b("latein"),*b("fremd"))))
    V.append(("V3","blass ~= original (Placebo haelt)", gleich(*b("blass"),k0,n0)))
    V.append(("V4","ohne_local < original",  faellt(*b("ohne_local"),k0,n0)))
    V.append(("V5","von ~= original",        gleich(*b("von"),k0,n0)))
    return V
def urteil_verhalten(V):
    """Reihenfolge = Schadenshoehe: was die Lesart am ehesten toetet, zuerst.
       V5 geht bewusst NICHT ins Urteil ein - es ist beschreibend."""
    d={n:ok for n,_,ok in V}
    if not d["V3"]: return "UNSPEZIFISCH"        # Placebo faellt mit -> Einfuegung wirkt
    if not (d["V1"] or d["V2"]): return "KEIN-ANKEREFFEKT"
    if (d["V1"]!=d["V2"]) or (not d["V2b"]): return "ANKER-UNEINIG"
    if not d["V4"]: return "LOCAL-NICHT-NOETIG"
    return "REFERENZLUECKE"
# ---------------- Ausfuehrung ------------------------------------------------
N_ARM=int(globals().get("N_ARM",48)); MAX_NEW=int(globals().get("MAX_NEW",40))
CHUNK=int(globals().get("CHUNK",16)); TEMP=float(globals().get("TEMP",1.0))
SEED=int(globals().get("SEED",20260805))
SCAFF="<|im_start|>user\n"
def prompt_text(u):
    """Denken aus - identisches Geruest wie Cell 29b"""
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
if not ZIEL_ID:
    _tr=[p for p in PROMPTS if PHRASE in PROMPTS[p]]
    assert _tr, "Zielprompt (Phrase %r) nicht im Korpus gefunden"%PHRASE
    ZIEL_ID=_tr[0]
BASIS_PROMPT=PROMPTS[ZIEL_ID]
assert BASIS_PROMPT.count(PHRASE)==1, ("Zielphrase kommt %dx vor - die Ersetzung "
    "waere nicht eindeutig"%BASIS_PROMPT.count(PHRASE))
print("="*74)
print("VERHALTENS-TEST DER REFERENZLUECKE | Prompt %s | %d Arme x %d Ziehungen"
      %(ZIEL_ID,len(ARME),N_ARM))
print("="*74)
print("Zielphrase: %r"%PHRASE)
TEXTE={}
for nm,ers,_ in ARME:
    t,ok=setze_arm(BASIS_PROMPT,ers)
    assert ok, "Ersetzung fuer %s fehlgeschlagen"%nm
    TEXTE[nm]=t
    print("  %-11s -> %r"%(nm,ers))
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
K={}; N={}; CLS={}; BSP={}
t0=time.time()
for ai,(nm,ers,kom) in enumerate(ARME):
    txt=prompt_text(TEXTE[nm]); cls=[]
    for b0 in range(0,N_ARM,CHUNK):
        b=min(CHUNK,N_ARM-b0)
        enc=tokenizer([txt]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(SEED+1009*ai+b0)
        with torch.no_grad():
            gen=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                               repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                               pad_token_id=tokenizer.pad_token_id)
        for j in range(b):
            a=tokenizer.decode(gen[j,enc["input_ids"].shape[1]:],skip_special_tokens=True)
            c=classify_answer(a); cls.append(c)
            if c in SW and nm not in BSP: BSP[nm]=a[:160]
    CLS[nm]=collections.Counter(cls)
    K[nm]=sum(1 for c in cls if c in SW); N[nm]=len(cls)
    p,lo,hi=wilson(K[nm],N[nm])
    print("  [%d/%d] %-11s %2d/%-2d = %5.1f%%  [%4.1f, %4.1f]  (%.0f s)"
          %(ai+1,len(ARME),nm,K[nm],N[nm],100*p,100*lo,100*hi,time.time()-t0))
# ---------------- Kalibrier-Tor ----------------------------------------------
if K["original"]==0:
    print("")
    print("ABBRUCH-WARNUNG: die Baseline kippt in diesem Protokoll kein einziges Mal")
    print("(0/%d). Dann hat der Test keine Kraft und keine Zahl unten ist deutbar."%N_ARM)
    print("Ursache pruefen: Temperatur %.2f, MAX_NEW=%d, Denken aus?"%(TEMP,MAX_NEW))
# ---------------- Auswertung -------------------------------------------------
print("")
print("KIPPRATEN (kippen = takeover | gloss | latin-switch(fr), Wilson-95%)")
print("  %-11s %6s %7s %18s %9s   %s"%("Arm","k/n","Rate","95%-Intervall","p vs Basis","Deutung"))
k0,n0=K["original"],N["original"]
for nm,ers,kom in ARME:
    p,lo,hi=wilson(K[nm],N[nm])
    pv=twoprop(K[nm],N[nm],k0,n0) if nm!="original" else float("nan")
    print("  %-11s %2d/%-3d %6.1f%% [%5.1f%%, %5.1f%%] %9s   %s"
          %(nm,K[nm],N[nm],100*p,100*lo,100*hi,
            "-" if nm=="original" else "%.4f"%pv,kom))
print("")
print("KLASSEN je Arm (empty ist ein Warnsignal, nicht ein Ergebnis):")
for nm,_,_ in ARME:
    print("  %-11s %s"%(nm," ".join("%s=%d"%(c,n) for c,n in CLS[nm].most_common())))
print("")
print("VORAB REGISTRIERTE VORHERSAGEN:")
V=pruefe_vorhersagen(K,N)
for n_,txt,ok in V: print("  %-4s %-34s %s"%(n_,txt,"HAELT" if ok else "FAELLT"))
CODE=urteil_verhalten(V)
print("")
print("VERDIKT: %s"%CODE)
if CODE=="REFERENZLUECKE":
    print("  Beide Ortsanker senken die Kipprate, das blasse Adjektiv nicht, und")
    print("  ohne ' local' verschwindet sie. Das Wort oeffnet eine Referenzluecke;")
    print("  wer sie schliesst - egal in welcher Schrift - schliesst das Verhalten.")
elif CODE=="UNSPEZIFISCH":
    print("  Auch das blasse Adjektiv senkt die Rate. Dann wirkt die blosse")
    print("  Einfuegung, nicht der Ort. Die Anker-Deutung ist damit erledigt.")
elif CODE=="LOCAL-NICHT-NOETIG":
    print("  Ohne ' local' kippt es genauso oft. Dann traegt nicht das Wort die")
    print("  Luecke, sondern die Konstruktion - siehe Arme von/artikel.")
elif CODE=="KEIN-ANKEREFFEKT":
    print("  Kein Ortsanker senkt die Rate messbar. Die Dispositions-Messung aus")
    print("  Anker v2 hat sich dann nicht in Verhalten uebersetzt.")
elif CODE=="ANKER-UNEINIG":
    print("  Die beiden Ortsanker wirken nicht gleich stark. Genau das hatte die")
    print("  Dispositions-Messung ausgeschlossen - dann lebt die Schrift-Lesart")
    print("  im Verhalten weiter, auch wenn sie in der Disposition tot war.")
print("  V5 (von-Konstruktion) geht bewusst nicht ins Urteil ein: sie beschreibt,")
print("  ob der Genitiv oder das Wort die Luecke traegt.")
print("")
print("BEISPIELE (erste gekippte Antwort je Arm, gekuerzt):")
for nm,_,_ in ARME:
    if nm in BSP: print("  %-11s %r"%(nm,BSP[nm]))
    else:         print("  %-11s (keine gekippte Antwort)"%nm)
print("")
print("(Ein Zielprompt, %d Ziehungen je Arm, Temperatur %.2f, %d neue Token, Denken aus."
      %(N_ARM,TEMP,MAX_NEW))
print(" Alle Arme im selben Lauf - die Baseline ist die eigene, keine aus einem")
print(" frueheren Protokoll. Absolute Raten sind daher nicht mit frueheren Zahlen")
print(" vergleichbar, die Unterschiede zwischen den Armen sind es.)")
VERHALTEN_RESULTS=dict(verdict=CODE,prompt_id=ZIEL_ID,phrase=PHRASE,n_arm=N_ARM,
    max_new=MAX_NEW,temp=TEMP,seed=SEED,arme=[a[0] for a in ARME],
    ersatz={a[0]:a[1] for a in ARME},k={n:K[n] for n in K},n={n:N[n] for n in N},
    klassen={n:dict(CLS[n]) for n in CLS},
    p_vs_basis={n:(None if n=="original" else twoprop(K[n],N[n],k0,n0)) for n in K},
    vorhersagen=[dict(name=a,text=b,haelt=bool(c)) for a,b,c in V],
    beispiele=BSP)
wc_save_all()
